# 📊 Previsão da Taxa Selic

**Objetivo:** Pipeline completo de extração, análise e previsão da Taxa Selic Meta usando indicadores macroeconômicos e modelos de Machine Learning.

| Item | Detalhe |
|------|--------|
| **Horizonte** | 6 meses à frente |
| **Período** | 2020 – presente |
| **Modelos** | SARIMAX · Random Forest · XGBoost |
| **Fontes** | BCB/SGS · Yahoo Finance |

---

### Sumário

1. [Setup](#1) · 2. [Extração de Dados](#2) · 3. [Análise Exploratória](#3) · 4. [Feature Engineering](#4) · 5. [Modelagem SARIMAX](#5) · 6. [Modelagem ML](#6) · 7. [Comparação](#7) · 8. [Previsão Final](#8)

<a id='1'></a>
## 1. Setup & Configuração

In [ ]:
# Instalar dependências
!pip install python-bcb yfinance pandas numpy plotly matplotlib seaborn \
             statsmodels scikit-learn xgboost pmdarima shap -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from bcb import sgs
import yfinance as yf

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pmdarima import auto_arima

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor
import shap

# --- Parâmetros globais ---
START_DATE = '2020-01-01'
HORIZONTE  = 6  # meses à frente
TEST_SIZE  = 0.2
SEED       = 42

# --- Estilo ---
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (14, 5), 'font.size': 11})
TPL = 'plotly_dark'  # template plotly

print(f'✅ Setup OK — {pd.Timestamp.now():%Y-%m-%d %H:%M}')

<a id='2'></a>
## 2. Extração de Dados

### 2.1 Séries do SGS (Banco Central)

In [ ]:
# Códigos das séries no Sistema Gerenciador de Séries (SGS) do BCB
SERIES_SGS = {
    'selic_meta':           4189,   # Taxa Selic Meta definida pelo COPOM (% a.a.) — TARGET
    'ipca_mensal':          433,    # IPCA variação mensal (%)
    'ipca_acum_12m':        13522,  # IPCA acumulado 12 meses (%)
    'igpm_mensal':          189,    # IGP-M variação mensal (%)
    'cambio_usd':           3698,   # Câmbio USD/BRL (venda)
    'divida_liquida_pib':   4513,   # Dívida líquida / PIB (%)
    'resultado_primario':   4505,   # Resultado primário / PIB (%)
    'desemprego':           24369,  # Desocupação PNAD (%)
    'producao_industrial':  21859,  # Produção industrial (var. mensal %)
}

print(f'📋 {len(SERIES_SGS)} séries do SGS para extrair\n')

dfs = {}
for nome, codigo in SERIES_SGS.items():
    try:
        dfs[nome] = sgs.get({nome: codigo}, start=START_DATE)
        print(f'  ✅ {nome:<25} (cód. {codigo:>5}) — {len(dfs[nome]):>4} registros')
    except Exception as e:
        print(f'  ❌ {nome:<25} (cód. {codigo:>5}) — {e}')

df_sgs = pd.concat(dfs.values(), axis=1)
df_sgs.columns = dfs.keys()
df_sgs = df_sgs.resample('ME').last()

print(f'\n📊 SGS consolidado: {df_sgs.shape}')

### 2.2 Dados de Mercado (Yahoo Finance)

In [ ]:
tickers = {
    'petroleo_brent': 'BZ=F',
    'dxy_dollar':     'DX-Y.NYB',
}

dfs_yf = {}
for nome, ticker in tickers.items():
    try:
        tmp = yf.download(ticker, start=START_DATE, progress=False)
        col = 'Adj Close' if 'Adj Close' in tmp.columns else 'Close'
        serie = tmp[col]
        if isinstance(serie, pd.DataFrame):
            serie = serie.iloc[:, 0]
        dfs_yf[nome] = serie.rename(nome)
        print(f'  ✅ {nome:<20} ({ticker}) — {len(serie)} registros')
    except Exception as e:
        print(f'  ❌ {nome:<20} — {e}')

df_yf = pd.DataFrame(dfs_yf).resample('ME').last()
print(f'\n📊 Yahoo Finance: {df_yf.shape}')

### 2.3 Consolidação do Dataset

In [ ]:
df = pd.concat([df_sgs, df_yf], axis=1)
df = df.dropna(subset=['selic_meta'])  # manter apenas meses com Selic
df = df.ffill()  # forward-fill para indicadores com freq. menor

print(f'📦 Dataset consolidado: {df.shape}')
print(f'📅 Período: {df.index.min():%Y-%m} → {df.index.max():%Y-%m}')
print(f'❓ Nulos: {df.isnull().sum().sum()}')
df.describe().round(2)

<a id='3'></a>
## 3. Análise Exploratória (EDA)

### 3.1 Evolução Histórica da Selic

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df.index, y=df['selic_meta'],
    mode='lines', name='Selic Meta',
    line=dict(color='#00d4ff', width=2.5),
    fill='tozeroy', fillcolor='rgba(0,212,255,0.08)',
))

# Eventos marcantes
for data, texto, yp in [
    ('2020-08-01', 'Mínima histórica\nCOVID', 2.0),
    ('2022-08-01', 'Pico pós-COVID',        13.75),
]:
    if pd.Timestamp(data) >= df.index.min():
        fig.add_annotation(
            x=data, y=yp, text=texto, showarrow=True, arrowhead=2,
            arrowcolor='#ff6b6b', font=dict(size=10, color='white'),
            bgcolor='rgba(255,107,107,0.3)', borderpad=4,
        )

fig.update_layout(
    title='Evolução da Taxa Selic Meta (% a.a.)',
    xaxis_title='Data', yaxis_title='% a.a.',
    template=TPL, height=450, hovermode='x unified',
)
fig.show()

### 3.2 Painel de Indicadores

In [ ]:
paineis = [
    ('selic_meta',      'Selic Meta (%)',  '#00d4ff'),
    ('ipca_acum_12m',   'IPCA 12m (%)',    '#ff6b6b'),
    ('cambio_usd',      'Câmbio USD/BRL',  '#ffd93d'),
    ('desemprego',      'Desemprego (%)',   '#6bcb77'),
]
paineis = [(c, n, cor) for c, n, cor in paineis if c in df.columns]

fig = make_subplots(rows=2, cols=2, subplot_titles=[n for _, n, _ in paineis])
for i, (col, _, cor) in enumerate(paineis):
    fig.add_trace(
        go.Scatter(x=df.index, y=df[col], line=dict(color=cor, width=2),
                   fill='tozeroy', showlegend=False),
        row=i//2+1, col=i%2+1,
    )

fig.update_layout(title='Painel — Indicadores Macroeconômicos',
                  template=TPL, height=600)
fig.show()

### 3.3 Matriz de Correlação

In [ ]:
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=.5,
            cbar_kws={'shrink': .8, 'label': 'Pearson'}, ax=ax)
ax.set_title('Matriz de Correlação — Indicadores Macroeconômicos', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

print('\nCorrelação com Selic Meta (ordenada):')
print(corr['selic_meta'].drop('selic_meta').sort_values(ascending=False).round(3).to_string())

### 3.4 Análise de Defasagem (Lead-Lag)

In [ ]:
lag_cols = [c for c in ['ipca_acum_12m','cambio_usd','desemprego','igpm_mensal']
           if c in df.columns]
MAX_LAG = 12

fig, axes = plt.subplots(1, len(lag_cols), figsize=(5*len(lag_cols), 4))
if len(lag_cols) == 1: axes = [axes]

for ax, col in zip(axes, lag_cols):
    idx = df[['selic_meta', col]].dropna().index
    s1 = ((df.loc[idx, 'selic_meta'] - df.loc[idx, 'selic_meta'].mean())
          / df.loc[idx, 'selic_meta'].std()).values
    s2 = ((df.loc[idx, col] - df.loc[idx, col].mean())
          / df.loc[idx, col].std()).values

    lags = range(-MAX_LAG, MAX_LAG+1)
    cc = [np.corrcoef(s1[MAX_LAG:-MAX_LAG],
                      np.roll(s2, l)[MAX_LAG:-MAX_LAG])[0,1] for l in lags]

    cores = ['#1a73e8' if v >= 0 else '#d93025' for v in cc]
    ax.bar(lags, cc, color=cores, alpha=.7)
    best = list(lags)[np.argmax(np.abs(cc))]
    ax.axvline(best, color='#e8710a', ls='--', alpha=.8)
    ax.set_title(f'{col}  (melhor lag: {best}m)', fontsize=11)
    ax.set_xlabel('Lag (meses)'); ax.set_ylabel('Correlação')

plt.suptitle('Análise de Defasagem vs Selic', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

### 3.5 Decomposição da Série

In [ ]:
selic_series = df['selic_meta'].dropna()
if len(selic_series) >= 24:  # precisa de pelo menos 2 ciclos
    dec = seasonal_decompose(selic_series, model='additive', period=12)

    comps = [
        (dec.observed, 'Observado', '#1a73e8'),
        (dec.trend,    'Tendência', '#e8710a'),
        (dec.seasonal, 'Sazonalidade', '#0d904f'),
        (dec.resid,    'Resíduo', '#d93025'),
    ]

    fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
    for ax, (data, titulo, cor) in zip(axes, comps):
        ax.plot(data, color=cor, lw=1.5)
        ax.set_ylabel(titulo); ax.grid(alpha=.3)
    axes[0].set_title('Decomposição — Selic Meta', fontsize=14)
    plt.tight_layout(); plt.show()
else:
    print('⚠️ Série muito curta para decomposição sazonal (mín. 24 meses)')

<a id='4'></a>
## 4. Feature Engineering

Criação de features derivadas para capturar dinâmicas temporais.

In [ ]:
df_feat = df.copy()
feat_base = [c for c in df_feat.columns if c != 'selic_meta']

# --- 4.1  Features da Selic ---
df_feat['selic_diff']     = df_feat['selic_meta'].diff()
df_feat['selic_direcao']  = np.sign(df_feat['selic_diff'])
df_feat['selic_dist_mm12'] = df_feat['selic_meta'] - df_feat['selic_meta'].rolling(12).mean()

# Duração do ciclo (meses consecutivos na mesma direção)
ciclo, cnt, prev = [], 0, 0
for d in df_feat['selic_direcao'].fillna(0):
    cnt = cnt + 1 if (d == prev and d != 0) else (1 if d != 0 else 0)
    prev = d; ciclo.append(cnt)
df_feat['selic_ciclo'] = ciclo

# --- 4.2  Lags (ajustados para janela menor) ---
for col in feat_base:
    for lag in [1, 3, 6]:
        df_feat[f'{col}_L{lag}'] = df_feat[col].shift(lag)

# --- 4.3  Médias Móveis ---
for col in feat_base:
    for w in [3, 6]:
        df_feat[f'{col}_MM{w}'] = df_feat[col].rolling(w).mean()

# --- 4.4  Variações ---
for col in ['selic_meta', 'cambio_usd', 'petroleo_brent']:
    if col in df_feat.columns:
        df_feat[f'{col}_MoM'] = df_feat[col].pct_change(1)  * 100
        df_feat[f'{col}_YoY'] = df_feat[col].pct_change(12) * 100

# --- 4.5  Target ---
TARGET = f'target_selic_{HORIZONTE}m'
df_feat[TARGET] = df_feat['selic_meta'].shift(-HORIZONTE)

# Limpar NaNs
n_antes = len(df_feat)
df_feat = df_feat.dropna(subset=[TARGET]).ffill().bfill().dropna()

print(f'✅ Features criadas: {df_feat.shape[1] - df.shape[1]}')
print(f'📊 Shape final: {df_feat.shape}  ({n_antes - len(df_feat)} linhas removidas)')

In [ ]:
# --- 4.6  Teste de Estacionariedade (ADF) nas séries originais ---
print(f'{"Série":<25} {"Estatística":>12} {"p-valor":>10} {"Estacionária":>14}')
print('-' * 63)

for col in df.columns:
    s = df[col].dropna()
    if len(s) < 20: continue
    stat, pval = adfuller(s, autolag='AIC')[:2]
    ok = '✅ Sim' if pval < 0.05 else '❌ Não'
    print(f'{col:<25} {stat:>12.4f} {pval:>10.4f} {ok:>14}')

<a id='5'></a>
## 5. Modelagem — SARIMAX

In [ ]:
# Variáveis exógenas (apenas originais, máx. 4 para estabilidade com janela curta)
exog_cols = [c for c in ['ipca_acum_12m','cambio_usd','desemprego','producao_industrial']
             if c in df_feat.columns][:4]

y_sar  = df_feat['selic_meta']
X_sar  = df_feat[exog_cols]
n_test = int(len(y_sar) * TEST_SIZE)

y_train_s, y_test_s = y_sar.iloc[:-n_test], y_sar.iloc[-n_test:]
X_train_s, X_test_s = X_sar.iloc[:-n_test], X_sar.iloc[-n_test:]

# Auto-ARIMA
print('🔍 Buscando melhor ordem (p,d,q)...')
auto_model = auto_arima(y_train_s, exogenous=X_train_s, seasonal=False,
                        stepwise=True, suppress_warnings=True,
                        max_p=3, max_d=2, max_q=3, information_criterion='aic')
ORDER = auto_model.order
print(f'✅ Melhor ordem: {ORDER}')

# Treinar SARIMAX
modelo_sar = SARIMAX(y_train_s, exog=X_train_s, order=ORDER,
                     enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
pred_sar = modelo_sar.forecast(steps=n_test, exog=X_test_s)

print(modelo_sar.summary().tables[1])

In [ ]:
# Gráfico SARIMAX
fig = go.Figure()
fig.add_trace(go.Scatter(x=y_train_s.index, y=y_train_s, name='Treino',
                         line=dict(color='#636efa', width=1.5)))
fig.add_trace(go.Scatter(x=y_test_s.index, y=y_test_s, name='Real',
                         line=dict(color='#00d4ff', width=2.5)))
fig.add_trace(go.Scatter(x=y_test_s.index, y=pred_sar.values, name='SARIMAX',
                         line=dict(color='#ff6b6b', width=2.5, dash='dash')))

fig.update_layout(title=f'SARIMAX{ORDER} — Previsão vs Real',
                  yaxis_title='Selic (%)', template=TPL, height=450,
                  hovermode='x unified')
fig.show()

<a id='6'></a>
## 6. Modelagem — Machine Learning

### 6.1 Preparação dos Dados

> ⚠️ **Split temporal** (nunca K-Fold aleatório em séries temporais!)

In [ ]:
# Separar features e target
target_cols = [c for c in df_feat.columns if c.startswith('target_')]
feature_cols = [c for c in df_feat.columns if c not in target_cols]

X = df_feat[feature_cols].values
y = df_feat[TARGET].values
dates = df_feat.index

n_test = int(len(X) * TEST_SIZE)
n_train = len(X) - n_test

scaler = StandardScaler()
X_train = scaler.fit_transform(X[:n_train])
X_test  = scaler.transform(X[n_train:])
y_train, y_test = y[:n_train], y[n_train:]
dates_train, dates_test = dates[:n_train], dates[n_train:]

print(f'Train: {n_train} ({dates_train[0]:%Y-%m} → {dates_train[-1]:%Y-%m})')
print(f'Test:  {n_test}  ({dates_test[0]:%Y-%m} → {dates_test[-1]:%Y-%m})')
print(f'Features: {len(feature_cols)}')

In [ ]:
# --- Função auxiliar de métricas ---
def metricas(y_true, y_pred, nome):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mask  = y_true != 0
    mape = np.mean(np.abs((y_true[mask]-y_pred[mask])/y_true[mask]))*100 if mask.any() else np.nan
    print(f'\n📊 {nome}:  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%  R²={r2:.4f}')
    return {'modelo': nome, 'mae': round(mae,4), 'rmse': round(rmse,4),
            'mape': round(mape,2), 'r2': round(r2,4)}

### 6.2 Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=8,
                           random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
met_rf  = metricas(y_test, pred_rf, 'Random Forest')

imp_rf = (pd.DataFrame({'feature': feature_cols, 'imp': rf.feature_importances_})
          .sort_values('imp', ascending=False))
print('\nTop 10 features:')
print(imp_rf.head(10).to_string(index=False))

### 6.3 XGBoost

In [ ]:
xgb = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05,
                   random_state=SEED, n_jobs=-1, verbosity=0)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
pred_xgb = xgb.predict(X_test)
met_xgb  = metricas(y_test, pred_xgb, 'XGBoost')

imp_xgb = (pd.DataFrame({'feature': feature_cols, 'imp': xgb.feature_importances_})
           .sort_values('imp', ascending=False))
print('\nTop 10 features:')
print(imp_xgb.head(10).to_string(index=False))

### 6.4 Validação Temporal (TimeSeriesSplit)

In [ ]:
X_full = np.vstack([X_train, X_test])
y_full = np.concatenate([y_train, y_test])
tscv   = TimeSeriesSplit(n_splits=4)

print('Validação Temporal — XGBoost (4 folds)')
print('-' * 55)

fold_results = []
for fold, (tr, te) in enumerate(tscv.split(X_full)):
    m = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05,
                     random_state=SEED, verbosity=0)
    m.fit(X_full[tr], y_full[tr])
    p = m.predict(X_full[te])
    mae  = mean_absolute_error(y_full[te], p)
    rmse = np.sqrt(mean_squared_error(y_full[te], p))
    fold_results.append({'fold': fold+1, 'mae': mae, 'rmse': rmse})
    print(f'  Fold {fold+1}: MAE={mae:.4f}  RMSE={rmse:.4f}  (train={len(tr)}, test={len(te)})')

df_folds = pd.DataFrame(fold_results)
print('-' * 55)
print(f'  Média:  MAE={df_folds.mae.mean():.4f}  RMSE={df_folds.rmse.mean():.4f}')
print(f'  Desvio: MAE={df_folds.mae.std():.4f}  RMSE={df_folds.rmse.std():.4f}')

### 6.5 SHAP — Interpretabilidade

In [ ]:
explainer   = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(shap_values, features=X_test,
                  feature_names=feature_cols, max_display=20, show=False)
plt.title('SHAP Values — Top 20 Features (XGBoost)', fontsize=14)
plt.tight_layout(); plt.show()

<a id='7'></a>
## 7. Avaliação & Comparação

In [ ]:
# Métricas SARIMAX
met_sar = metricas(y_test_s.values, pred_sar.values, 'SARIMAX')

# Tabela comparativa
df_comp = pd.DataFrame([met_sar, met_rf, met_xgb])
print('\n' + '='*60)
print('  COMPARAÇÃO DE MODELOS')
print('='*60)
print(df_comp.to_string(index=False))
melhor = df_comp.loc[df_comp.mae.idxmin(), 'modelo']
print(f'\n🏆 Melhor modelo (MAE): {melhor}')

In [ ]:
# Gráfico comparativo de métricas
fig = make_subplots(rows=1, cols=3, subplot_titles=['MAE (p.p.)', 'RMSE (p.p.)', 'R²'])
cores = ['#636efa', '#00cc96', '#ffa15a']

for i, met in enumerate(['mae', 'rmse', 'r2']):
    fig.add_trace(
        go.Bar(x=df_comp['modelo'], y=df_comp[met], marker_color=cores,
               text=df_comp[met], textposition='outside', showlegend=False),
        row=1, col=i+1,
    )

fig.update_layout(title='Comparação de Modelos', template=TPL, height=400)
fig.show()

In [ ]:
# Previsão de todos os modelos vs real
fig = go.Figure()

fig.add_trace(go.Scatter(x=dates_test, y=y_test, name='Real',
                         line=dict(color='white', width=3), mode='lines+markers',
                         marker=dict(size=4)))
fig.add_trace(go.Scatter(x=dates_test, y=pred_rf,  name='Random Forest',
                         line=dict(color='#00cc96', width=2, dash='dot')))
fig.add_trace(go.Scatter(x=dates_test, y=pred_xgb, name='XGBoost',
                         line=dict(color='#ffa15a', width=2, dash='dash')))

fig.update_layout(title=f'Previsão Selic em t+{HORIZONTE}m — Modelos vs Real',
                  yaxis_title=f'Selic t+{HORIZONTE}m (%)',
                  template=TPL, height=450, hovermode='x unified',
                  legend=dict(x=.01, y=.99))
fig.show()

In [ ]:
# Feature Importance — RF vs XGBoost
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, (imp, nome, cor) in zip(axes, [(imp_rf,'Random Forest','#00cc96'),
                                       (imp_xgb,'XGBoost','#ffa15a')]):
    top = imp.head(15)
    ax.barh(range(len(top)), top['imp'].values, color=cor, alpha=.8)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top['feature'].values, fontsize=10)
    ax.set_xlabel('Importância')
    ax.set_title(f'Top 15 — {nome}', fontsize=13)
    ax.invert_yaxis()

plt.tight_layout(); plt.show()

<a id='8'></a>
## 8. Previsão Final & Conclusão

### 8.1 Projeção para os próximos 6 meses

In [ ]:
# Selecionar melhor modelo
modelo_final = xgb if melhor == 'XGBoost' else rf
X_ultimo = scaler.transform(df_feat[feature_cols].iloc[-1:].values)
previsao = modelo_final.predict(X_ultimo)[0]

selic_atual = df['selic_meta'].iloc[-1]
datas_futuras = pd.date_range(df.index[-1] + pd.DateOffset(months=1),
                              periods=HORIZONTE, freq='ME')

df_prev = pd.DataFrame({
    'data': datas_futuras,
    'cenario_otimista':  previsao - 0.5,
    'cenario_base':      previsao,
    'cenario_pessimista': previsao + 0.5,
})

print(f'Selic atual:          {selic_atual:.2f}%')
print(f'Previsão ({HORIZONTE}m):     {previsao:.2f}%')
print(f'Cenários:             [{previsao-.5:.2f}%  |  {previsao:.2f}%  |  {previsao+.5:.2f}%]')
df_prev

In [ ]:
# Gráfico final: histórico + projeção com banda de confiança
fig = go.Figure()

# Histórico completo
fig.add_trace(go.Scatter(x=df.index, y=df['selic_meta'], name='Selic Real',
                         line=dict(color='#00d4ff', width=2.5)))

# Previsão
fig.add_trace(go.Scatter(x=df_prev['data'], y=df_prev['cenario_base'],
                         name='Previsão (base)', mode='lines+markers',
                         line=dict(color='#ffa15a', width=2.5, dash='dash')))

# Banda otimista/pessimista
fig.add_trace(go.Scatter(
    x=pd.concat([df_prev['data'], df_prev['data'][::-1]]),
    y=pd.concat([df_prev['cenario_otimista'], df_prev['cenario_pessimista'][::-1]]),
    fill='toself', fillcolor='rgba(255,161,90,0.15)',
    line=dict(color='rgba(255,255,255,0)'), name='Banda ±0.5 p.p.',
))

fig.update_layout(title=f'Projeção da Selic — Próximos {HORIZONTE} Meses ({melhor})',
                  yaxis_title='Selic (%)', template=TPL, height=450,
                  hovermode='x unified')
fig.show()

### 8.2 Conclusões

**Principais achados:**

1. **Inflação é o driver principal** — IPCA acumulado 12m e suas defasagens são as features mais importantes em todos os modelos, confirmando o regime de metas de inflação.

2. **Defasagem de 1-3 meses** — Os indicadores macroeconômicos "lideram" as decisões do COPOM com delay típico de 1 a 3 meses.

3. **XGBoost vs SARIMAX** — O XGBoost captura melhor as não-linearidades da política monetária; o SARIMAX captura a dinâmica temporal de curto prazo.

4. **SHAP confirma teoria** — As features mais relevantes (inflação, câmbio, expectativas) são consistentes com os fundamentos que o COPOM considera em suas decisões.

**Limitações:**
- A Selic é definida por decisão de comitê (componentes discricionários e políticos)
- Eventos extraordinários (crises, pandemias) quebram padrões históricos
- Para horizontes longos a incerteza cresce exponencialmente
- Janela de dados curta (2020+) pode limitar a generalização

**Próximos passos:**
- Incluir NLP das atas do COPOM como feature
- Testar ensemble/stacking dos modelos
- Backtesting com rolling-window
- Dashboard interativo com Streamlit

In [ ]:
print('\n' + '='*60)
print('  ✅ ANÁLISE CONCLUÍDA')
print('='*60)
print(f'  📅 {pd.Timestamp.now():%Y-%m-%d %H:%M}')
print(f'  📊 {df_feat.shape[0]} meses × {df_feat.shape[1]} features')
print(f'  🎯 Horizonte: {HORIZONTE} meses')
print(f'  🏆 Melhor modelo: {melhor}')
print('='*60)